In [1]:
import numpy as np, matplotlib.pyplot as plt, pandas as pd

pd.set_option("display.max_rows", 8)
!date

Sat Aug 31 21:57:10 PDT 2024


# Mean deaths and stillbirths averted by adding folate by wealth quintile


In [2]:
import vivarium_inputs
import db_queries
import gbd_mapping
import pathlib
from lsff_utils import config_utils

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [3]:
location = "india"
vehicle = "rice"

In [4]:
# Parameters
location = "india"
vehicle = "rice"

In [5]:
intervention_scenarios = config_utils.get_config()["custom_intervention_scenarios"].get(
    location, ["intervention"]
)
intervention_scenarios

['intervention']

## Forecasted births and stillbirths

In [6]:
asfr = vivarium_inputs.get_measure(
    gbd_mapping.covariates.age_specific_fertility_rate,
    "estimate",
    location.title(),
    years=2022,
).value

In [7]:
asfr = asfr[asfr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
)

In [8]:
# Scale ASFR in each category down proportionally to the scale-down in TFR forecasted from GBD 2017
if location == "india":
    asfr_2030_to_2022_ratio = 1.61 / 1.91  # http://ihmeuw.org/6j8s
elif location == "nigeria":
    asfr_2030_to_2022_ratio = 4.43 / 4.96  # http://ihmeuw.org/6jqx
elif location == "ethiopia":
    asfr_2030_to_2022_ratio = 3.27 / 4.10  # http://ihmeuw.org/6j7d

asfr = asfr * asfr_2030_to_2022_ratio
asfr[asfr > 0]

location  sex     age_start  age_end  year_start  year_end
India     Female  10.0       15.0     2022        2023        0.000315
                  15.0       20.0     2022        2023        0.008079
                  20.0       25.0     2022        2023        0.087454
                  25.0       30.0     2022        2023        0.110737
                                                                ...   
                  35.0       40.0     2022        2023        0.028718
                  40.0       45.0     2022        2023        0.009527
                  45.0       50.0     2022        2023        0.002772
                  50.0       55.0     2022        2023        0.000257
Name: value, Length: 9, dtype: float64

In [9]:
asfr = (
    asfr.reset_index()
    .assign(year_start=2030, year_end=2031)
    .set_index(asfr.index.names)
    .value
)
asfr.sort_values()

location  sex     age_start  age_end    year_start  year_end
India     Female  0.000000   0.019178   2030        2031        0.000000
                  0.019178   0.076712   2030        2031        0.000000
                  0.076712   0.500000   2030        2031        0.000000
                  0.500000   1.000000   2030        2031        0.000000
                                                                  ...   
                  35.000000  40.000000  2030        2031        0.028718
                  30.000000  35.000000  2030        2031        0.069523
                  20.000000  25.000000  2030        2031        0.087454
                  25.000000  30.000000  2030        2031        0.110737
Name: value, Length: 50, dtype: float64

In [10]:
from vivarium_inputs import utilities
from vivarium_inputs.utility_data import get_location_id
from vivarium_gbd_access.gbd import get_age_group_id, SEX, RELEASE_IDS

In [11]:
def get_population_future(location, year):
    # Cobbled together from pieces of vivarium_inputs and vivarium_gbd_access
    # TODO: vivarium_inputs should be able to get forecasted pop!
    location_id = get_location_id(location)
    year_id = year
    data = db_queries.get_population(
        age_group_id=get_age_group_id(),
        location_id=location_id,
        year_id=year_id,
        sex_id=SEX.MALE + SEX.FEMALE + SEX.COMBINED,
        release_id=RELEASE_IDS.GBD_2021,
        forecasted_pop=True,
    )
    data = utilities.normalize_sex(
        data.drop("run_id", axis="columns").rename(columns={"population": "value"}),
        fill_value=None,
        cols_to_fill=utilities.DRAW_COLUMNS,
    )
    data = utilities.reshape(data, ["value"])
    data = utilities.scrub_gbd_conventions(data, location)
    data = utilities.split_interval(
        data, interval_column="age", split_column_prefix="age"
    )
    data = utilities.split_interval(
        data, interval_column="year", split_column_prefix="year"
    )
    return utilities.sort_hierarchical_data(data)

In [12]:
pop = get_population_future(location.title(), 2030).value.reindex(asfr.index)
pop

location  sex     age_start  age_end     year_start  year_end
India     Female  0.000000   0.019178    2030        2031        1.761904e+05
                  0.019178   0.076712    2030        2031        5.254291e+05
                  0.076712   0.500000    2030        2031                 NaN
                  0.500000   1.000000    2030        2031                 NaN
                                                                     ...     
          Male    80.000000  85.000000   2030        2031        6.771079e+06
                  85.000000  90.000000   2030        2031        2.880948e+06
                  90.000000  95.000000   2030        2031        9.755468e+05
                  95.000000  125.000000  2030        2031        2.694498e+05
Name: value, Length: 50, dtype: float64

In [13]:
# Forecasted population does not have younger ages, but luckily none of these are WRA
assert (pop.index.get_level_values("age_end")[pop.isna()] < 10).all()
pop[pop.isna()]

location  sex     age_start  age_end  year_start  year_end
India     Female  0.076712   0.5      2030        2031       NaN
                  0.500000   1.0      2030        2031       NaN
                  1.000000   2.0      2030        2031       NaN
                  2.000000   5.0      2030        2031       NaN
          Male    0.076712   0.5      2030        2031       NaN
                  0.500000   1.0      2030        2031       NaN
                  1.000000   2.0      2030        2031       NaN
                  2.000000   5.0      2030        2031       NaN
Name: value, dtype: float64

In [14]:
pop = pop.fillna(0)

In [15]:
n_births = (pop * asfr).sum()
n_births

19609719.34389394

In [16]:
sbr = vivarium_inputs.get_measure(
    gbd_mapping.covariates.stillbirth_to_live_birth_ratio,
    "estimate",
    location.title(),
    years=2022,
).value
sbr

location  year_start  year_end  parameter  
India     2022        2023      lower_value    0.016328
                                mean_value     0.016328
                                upper_value    0.016328
Name: value, dtype: float64

In [17]:
sbr = sbr[sbr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
)
sbr

location  year_start  year_end
India     2022        2023        0.016328
Name: value, dtype: float64

In [18]:
sbr = sbr.values[0]

In [19]:
births_and_stillbirths = n_births + n_births * sbr
births_and_stillbirths / 1e6

19.929907151540085

## Fertility (technically birth-and-stillbirth) disparities

In [20]:
if location == "india":
    dist_births_and_stillbirths_by_wealth = pd.Series(  # from file:///J:/DATA/DHS_PROG_DHS/IND/2019_2021/IND_DHS7_2019_2021_REP_FINAL_Y2022M05D11.PDF
        {  # Table 8.4 Perinatal mortality -- using "Number of pregnancies of 7 or more months' duration" as a proxy
            1: 56_979,
            2: 50_335,
            3: 45_189,
            4: 42_611,
            5: 36_290,
        }
    )
elif location == "nigeria":
    dist_births_and_stillbirths_by_wealth = pd.Series(  # from file:///J:/DATA/DHS_PROG_DHS/NGA/2018/NGA_DHS7_2018_REP_QUEST_Y2019M11D05.PDF
        {  # Table 8.4 Perinatal mortality
            1: 7_712,
            2: 7_886,
            3: 7_139,
            4: 6_328,
            5: 5_558,
        }
    )
elif location == "ethiopia":
    dist_births_and_stillbirths_by_wealth = pd.Series(  # from file:///J:/DATA/DHS_PROG_DHS/ETH/2016/ETH_DHS7_2016_REP_QUEST_Y2017M08D15.PDF
        {  # Table 8.4 Perinatal mortality
            1: 2_645,
            2: 2_516,
            3: 2_290,
            4: 2_018,
            5: 1_592,
        }
    )

dist_births_and_stillbirths_by_wealth.index.name = "wealth_quintile"

s_births = (
    n_births
    * dist_births_and_stillbirths_by_wealth
    / dist_births_and_stillbirths_by_wealth.sum()
)
s_births

wealth_quintile
1    4.828535e+06
2    4.265506e+06
3    3.829422e+06
4    3.610956e+06
5    3.075300e+06
dtype: float64

In [21]:
s_births_and_stillbirths_by_wealth = (
    births_and_stillbirths
    * dist_births_and_stillbirths_by_wealth
    / dist_births_and_stillbirths_by_wealth.sum()
)
s_births_and_stillbirths_by_wealth

wealth_quintile
1    4.907375e+06
2    4.335154e+06
3    3.891949e+06
4    3.669916e+06
5    3.125514e+06
dtype: float64

In [22]:
# http://ihmeuw.org/6jr2 -- extracted from GBD Foresight, count of NTD deaths for under-1 year olds
if location == "india":
    ntd_deaths = 4_273.37
elif location == "nigeria":
    ntd_deaths = 5_373.52
elif location == "ethiopia":
    ntd_deaths = 1_883.76


ntd_death_rate = ntd_deaths / n_births
10_000 * ntd_death_rate

2.1792101789211165

In [23]:
# Assumed does not vary by wealth
champs_ntd_stillbirth_per_livebirth = 51 / (69 - 51)
ntd_stillbirths = ntd_deaths * champs_ntd_stillbirth_per_livebirth
10_000 * (
    ntd_deaths + ntd_stillbirths
) / n_births  # ntd rate, compare with 41 per 10,000 from Bhide et al https://pubmed.ncbi.nlm.nih.gov/23873811/

8.353639019197614

We could not find a good source for folate intake by wealth in India, or even a representative source for overall folate intake.  [This paper](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10755415/pdf/S1368980023002112a.pdf) has an overall number of 220 mcg/day for women, and since it is not too different from the values we found in Ethiopia and Nigeria, we are going to use it for now. (It also has standard deviation of 50, which we can use when we introduce heterogeneity)

In [24]:
if location == "india":
    s_baseline_folate = pd.Series(
        {
            1: 220,
            2: 220,
            3: 220,
            4: 220,
            5: 220,  # NRV is 400 mcg/day
        }
    )
elif location == "nigeria":
    # Table 95 of NFCMS 2021 Report
    s_baseline_folate = pd.Series(
        {
            1: 189,
            2: 198,
            3: 197,
            4: 203,
            5: 208,  # NRV is 400 mcg/day
        }
    )  # how can we use this to estimate disparities in NTD?
elif location == "ethiopia":
    # Table 6 of https://cdn.nutrition.org/article/S2475-2991%2824%2901728-1/fulltext
    s_baseline_folate = pd.Series(
        {
            1: 166,
            2: 152,
            3: 137,
            4: 350,
            5: 469,  # NRV is 400 mcg/day
        }
    )  # how can we use this to estimate disparities in NTD?

s_baseline_folate.index.name = "wealth_quintile"

In [25]:
if location == "india":
    s_dist_deaths_by_wealth = pd.Series(  # Table 7.9 on page 201 of the CNNS report has RBC folate deficiency rates;
        {  # it includes wealth stratification, but has a very low threshold for insufficiency
            1: 1,  # so I am assuming that most everyone is in the danger zone for low folate
            2: 1,
            3: 1,
            4: 1,
            5: 1,
        }
    )
elif location == "nigeria":
    s_dist_deaths_by_wealth = pd.Series(  # assume same rate for all, for now;
        {  # can CHAMPS offer more detail?  Need to infer wealth somehow
            1: 1,
            2: 1,
            3: 1,
            4: 1,
            5: 1,
        }
    )
elif location == "ethiopia":
    s_dist_deaths_by_wealth = pd.Series(
        {  # supplementation studies don't make this easy, but here is a guess
            1: 1,
            2: 1,
            3: 1,
            4: 1,
            5: 1,
        }
    )
    s_dist_deaths_by_wealth /= s_dist_deaths_by_wealth.mean()

s_dist_deaths_by_wealth.index.name = "wealth_quintile"
s_dist_deaths_by_wealth

wealth_quintile
1    1
2    1
3    1
4    1
5    1
dtype: int64

In [26]:
s_ntd_death_rate = ntd_deaths / n_births * s_dist_deaths_by_wealth
10_000 * s_ntd_death_rate

wealth_quintile
1    2.17921
2    2.17921
3    2.17921
4    2.17921
5    2.17921
dtype: float64

In [27]:
s_ntd_death_count = s_ntd_death_rate * s_births_and_stillbirths_by_wealth
s_ntd_death_count

wealth_quintile
1    1069.420132
2     944.721078
3     848.137495
4     799.751860
5     681.115088
dtype: float64

In [28]:
s_ntd_death_count.sum(), ntd_deaths  # should be similar

(4343.145652958891, 4273.37)

In [29]:
s_ntd_stillbirth_count = s_ntd_death_count * champs_ntd_stillbirth_per_livebirth
s_ntd_stillbirth_count

wealth_quintile
1    3030.023707
2    2676.709722
3    2403.056236
4    2265.963603
5    1929.826082
dtype: float64

In [30]:
s_ntd_death_or_stillbirth_count = s_ntd_death_count + s_ntd_stillbirth_count
s_ntd_death_or_stillbirth_count

wealth_quintile
1    4099.443838
2    3621.430801
3    3251.193731
4    3065.715463
5    2610.941169
dtype: float64

In [31]:
def backcalc_rbc(ntd_risk, method):
    """
    ln (odds of NTD risk) = 1.6563 − 1.2193 × ln (RBC) (Daly et al, 1995)
    ln (odds of NTD risk) = 4.57 − 1.70 × ln (RBC) (Crider et al, 2014)
    """

    odds = ntd_risk / (1 - ntd_risk)
    ln_odds = np.log(odds)
    if method == "daly":
        neg_ln_rbc = (ln_odds - 1.6563) / 1.2193
    elif method == "crider":
        neg_ln_rbc = (ln_odds - 4.57) / 1.70
    rbc = np.exp(-neg_ln_rbc)
    return rbc


backcalc_rbc(s_ntd_death_or_stillbirth_count / s_births, "daly")

wealth_quintile
1    1283.442766
2    1283.442766
3    1283.442766
4    1283.442766
5    1283.442766
dtype: float64

In [32]:
backcalc_rbc(
    s_ntd_death_or_stillbirth_count / s_births, "crider"
)  # compare with CNNS, https://www.unicef.org/india/media/2646/file/CNNS-report.pdf in Table 7.9

wealth_quintile
1    941.439754
2    941.439754
3    941.439754
4    941.439754
5    941.439754
dtype: float64

In [33]:
pop

location  sex     age_start  age_end     year_start  year_end
India     Female  0.000000   0.019178    2030        2031        1.761904e+05
                  0.019178   0.076712    2030        2031        5.254291e+05
                  0.076712   0.500000    2030        2031        0.000000e+00
                  0.500000   1.000000    2030        2031        0.000000e+00
                                                                     ...     
          Male    80.000000  85.000000   2030        2031        6.771079e+06
                  85.000000  90.000000   2030        2031        2.880948e+06
                  90.000000  95.000000   2030        2031        9.755468e+05
                  95.000000  125.000000  2030        2031        2.694498e+05
Name: value, Length: 50, dtype: float64

In [34]:
s_pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
s_pop = s_pop.set_index([c for c in s_pop.columns if c != "value"])
s_pop

value
sex    age_start age_end    pregnant     wealth_quintile              
Female 0.0       0.019178   not_pregnant 1                49649.700497
                                         2                43575.622144
                                         3                38689.356750
                                         4                36440.833935
...                                                                ...
Male   95.0      125.000000 not_pregnant 2                15848.509818
                                         3                16370.176208
                                         4                17265.313670
                                         5                21289.473456

[285 rows x 1 columns]

In [35]:
# WRA only
s_pop = s_pop[
    (s_pop.index.get_level_values("sex") == "Female")
    & (s_pop.index.get_level_values("age_start") >= 15)
    & (s_pop.index.get_level_values("age_end") <= 50)
].copy()

In [36]:
s_daily_vehicle = pd.read_csv(
    f"../0100_data_prep/results/{vehicle}/vehicle_consumption/amount/mean/{location}.csv"
)
assert (s_daily_vehicle.vehicle_name == vehicle).all()
s_daily_vehicle = s_daily_vehicle.drop(columns=["vehicle_name"])
s_daily_vehicle = s_daily_vehicle.set_index(
    [c for c in s_daily_vehicle.columns if c != "value"]
).value
s_daily_vehicle

sex     age_start  age_end  wealth_quintile
Female  0          5        1                  185.559631
                            2                  147.531394
                            3                  129.130302
                            4                  127.809856
                                                  ...    
Male    50         125      2                  185.190235
                            3                  176.087660
                            4                  170.708637
                            5                  133.137342
Name: value, Length: 50, dtype: float64

In [37]:
from lsff_utils import data_processing

In [38]:
s_daily_vehicle = (
    s_pop.value
    * data_processing.reindex_series_onto_df_by_age_groups(s_pop, s_daily_vehicle)
).groupby("wealth_quintile").sum() / s_pop.value.groupby("wealth_quintile").sum()
s_daily_vehicle

wealth_quintile
1    211.695632
2    174.262141
3    162.011968
4    162.387602
5    127.329933
Name: value, dtype: float64

In [39]:
any_consumers = pd.read_csv(
    f"../0100_data_prep/results/{vehicle}/vehicle_consumption/any/{location}.csv"
)
assert (any_consumers.vehicle_name == vehicle).all()
any_consumers = any_consumers.drop(columns=["vehicle_name"])
any_consumers = any_consumers.set_index(
    [c for c in any_consumers.columns if c != "value"]
).value
any_consumers

sex     age_start  age_end  wealth_quintile
Female  0          5        1                  0.919076
                            2                  0.887756
                            3                  0.875160
                            4                  0.874946
                                                 ...   
Male    50         125      2                  0.975934
                            3                  0.976151
                            4                  0.985469
                            5                  0.987381
Name: value, Length: 50, dtype: float64

In [40]:
any_consumers = (
    s_pop.value
    * data_processing.reindex_series_onto_df_by_age_groups(s_pop, any_consumers)
).groupby("wealth_quintile").sum() / s_pop.value.groupby("wealth_quintile").sum()
any_consumers

wealth_quintile
1    0.990320
2    0.971073
3    0.963712
4    0.985178
5    0.988109
Name: value, dtype: float64

In [41]:
s_daily_vehicle_among_consumers = s_daily_vehicle / any_consumers
s_daily_vehicle_among_consumers

wealth_quintile
1    213.764832
2    179.453099
3    168.112361
4    164.830721
5    128.862297
Name: value, dtype: float64

In [42]:
baseline_concentration_mcg_per_gram = pd.read_csv(
    f"../0100_data_prep/results/folate/{vehicle}/baseline_fortification/concentration/{location}.csv"
)
assert (baseline_concentration_mcg_per_gram.vehicle_name == vehicle).all()
assert baseline_concentration_mcg_per_gram.value.nunique() == 1
baseline_concentration_mcg_per_gram = baseline_concentration_mcg_per_gram.value.iloc[0]
baseline_concentration_mcg_per_gram

0.125

In [43]:
intervention_concentration_mcg_per_gram = pd.concat(
    [
        pd.read_csv(
            f"../0100_data_prep/results/folate/{vehicle}/{intervention_scenario}/intervention_fortification/concentration/{location}.csv"
        ).assign(scenario=intervention_scenario)
        for intervention_scenario in intervention_scenarios
    ]
)
assert (intervention_concentration_mcg_per_gram.vehicle_name == vehicle).all()
intervention_concentration_mcg_per_gram = (
    intervention_concentration_mcg_per_gram.drop(columns=["vehicle_name"])
    .set_index("scenario")
    .value
)
intervention_concentration_mcg_per_gram

scenario
intervention    1.3
Name: value, dtype: float64

In [44]:
eff_fort_baseline_path = f"../0100_data_prep/results/folate/{vehicle}/baseline_fortification/effective_coverage/{location}.csv"

df_eff_fort_baseline = pd.read_csv(eff_fort_baseline_path)
assert (df_eff_fort_baseline.vehicle_name == vehicle).all()
df_eff_fort_intervention = pd.concat(
    [
        pd.read_csv(
            f"../0100_data_prep/results/folate/{vehicle}/{intervention_scenario}/intervention_fortification/effective_coverage/{location}.csv"
        ).assign(scenario=intervention_scenario)
        for intervention_scenario in intervention_scenarios
    ]
)
assert (df_eff_fort_intervention.vehicle_name == vehicle).all()
df_eff_fort_intervention = df_eff_fort_intervention.drop(columns=["vehicle_name"])
df_eff_fort_intervention

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0,5,1,0.295892,intervention
1,Female,0,5,2,0.299771,intervention
2,Female,0,5,3,0.280152,intervention
3,Female,0,5,4,0.241038,intervention
...,...,...,...,...,...,...
46,Male,50,125,2,0.367314,intervention
47,Male,50,125,3,0.330535,intervention
48,Male,50,125,4,0.281447,intervention
49,Male,50,125,5,0.146075,intervention


In [45]:
# NOTE: Using DHS definition of WRA
population = (
    pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
    .groupby(["sex", "age_start", "age_end", "wealth_quintile"])
    .value.sum()
    .reset_index()
)
population = population[
    (population.sex == "Female")
    & (population.age_start >= 15)
    & (population.age_end <= 50)
]
population

,sex,age_start,age_end,wealth_quintile,value
40,Female,15.0,20.0,1,1.144883e+07
41,Female,15.0,20.0,2,1.300234e+07
42,Female,15.0,20.0,3,1.357840e+07
43,Female,15.0,20.0,4,1.349499e+07
...,...,...,...,...,...
71,Female,45.0,50.0,2,7.200663e+06
72,Female,45.0,50.0,3,7.650878e+06
73,Female,45.0,50.0,4,8.210394e+06
74,Female,45.0,50.0,5,8.760835e+06


In [46]:
if "sex" in df_eff_fort_baseline.columns:
    df_eff_fort_baseline = df_eff_fort_baseline[(df_eff_fort_baseline.sex == "Female")]

if "age_start" in df_eff_fort_baseline.columns:
    df_eff_fort_baseline = df_eff_fort_baseline[
        (df_eff_fort_baseline.age_start >= 15) & (df_eff_fort_baseline.age_end <= 50)
    ]

In [47]:
if "sex" in df_eff_fort_intervention.columns:
    df_eff_fort_intervention = df_eff_fort_intervention[
        (df_eff_fort_intervention.sex == "Female")
    ]

if "age_start" in df_eff_fort_intervention.columns:
    df_eff_fort_intervention = df_eff_fort_intervention[
        (df_eff_fort_intervention.age_start >= 15)
        & (df_eff_fort_intervention.age_end <= 50)
    ]

In [48]:
def aggregate_using_population(effective_fort):
    merge_cols = [
        c
        for c in ["sex", "wealth_quintile", "age_start", "age_end"]
        if c in effective_fort.columns
    ]
    merged = effective_fort.merge(
        population.reset_index(),
        on=[c for c in merge_cols if "age" not in c],
        suffixes=("_fort", "_pop"),
    )
    assert ("age_start" in merge_cols) == ("age_end" in merge_cols)
    if "age_start" in merge_cols:
        merged = merged[
            (merged.age_start_pop >= merged.age_start_fort)
            & (merged.age_end_pop <= merged.age_end_fort)
        ]
    print(merged)
    assert len(merged) == len(population) * (
        1
        if "scenario" not in effective_fort.columns
        else effective_fort.scenario.nunique()
    )
    group_cols = [c for c in ["scenario", "wealth_quintile"] if c in merged]
    return merged.groupby(group_cols).apply(
        lambda df: (df.value_fort * df.value_pop).sum() / df.value_pop.sum()
    )

In [49]:
df_eff_fort_baseline = aggregate_using_population(df_eff_fort_baseline)
df_eff_fort_baseline

    wealth_quintile vehicle_name     sex  age_start_fort  age_end_fort  \
0                 1         rice  Female              15            30   
1                 1         rice  Female              15            30   
2                 1         rice  Female              15            30   
10                1         rice  Female              30            50   
..              ...          ...     ...             ...           ...   
66                5         rice  Female              30            50   
67                5         rice  Female              30            50   
68                5         rice  Female              30            50   
69                5         rice  Female              30            50   

    value_fort  index  age_start_pop  age_end_pop     value_pop  
0     0.340944     40           15.0         20.0  1.144883e+07  
1     0.340944     45           20.0         25.0  1.161927e+07  
2     0.340944     50           25.0         30.0  1.097286e+

wealth_quintile
1    0.354760
2    0.357360
3    0.319993
4    0.282220
5    0.158611
dtype: float64

In [50]:
df_eff_fort_intervention = aggregate_using_population(df_eff_fort_intervention)
df_eff_fort_intervention

       sex  age_start_fort  age_end_fort  wealth_quintile  value_fort  \
0   Female              15            30                1    0.340944   
1   Female              15            30                1    0.340944   
2   Female              15            30                1    0.340944   
10  Female              30            50                1    0.368677   
..     ...             ...           ...              ...         ...   
66  Female              30            50                5    0.151782   
67  Female              30            50                5    0.151782   
68  Female              30            50                5    0.151782   
69  Female              30            50                5    0.151782   

        scenario  index  age_start_pop  age_end_pop     value_pop  
0   intervention     40           15.0         20.0  1.144883e+07  
1   intervention     45           20.0         25.0  1.161927e+07  
2   intervention     50           25.0         30.0  1.097286e+07

scenario      wealth_quintile
intervention  1                  0.354760
              2                  0.357360
              3                  0.319993
              4                  0.282220
              5                  0.158611
dtype: float64

In [51]:
RBC_baseline = backcalc_rbc(s_ntd_death_or_stillbirth_count / s_births, "crider")
RBC_baseline

wealth_quintile
1    941.439754
2    941.439754
3    941.439754
4    941.439754
5    941.439754
dtype: float64

In [52]:
# Fortification folate needs to be converted into dietary folate equivalents (DFEs)
# for use with our effect size.
# https://www.jandonline.org/article/S0002-8223(00)00027-4/pdf
fortification_mcg_to_dfe = 1.7

In [53]:
s_zero_folate = s_baseline_folate - (
    df_eff_fort_baseline
    * s_daily_vehicle_among_consumers
    * baseline_concentration_mcg_per_gram
    * fortification_mcg_to_dfe
)

In [54]:
s_intervention_folate = s_zero_folate + (
    df_eff_fort_intervention
    * s_daily_vehicle_among_consumers
    * intervention_concentration_mcg_per_gram
    * fortification_mcg_to_dfe
)
s_intervention_folate

scenario      wealth_quintile
intervention  1                  371.480943
              2                  348.098258
              3                  327.455129
              4                  312.920689
              5                  260.826859
dtype: float64

In [55]:
zero_folate_pct_decrease = (s_baseline_folate - s_zero_folate) / s_baseline_folate

In [56]:
intevention_folate_pct_increase_from_zero = (
    s_intervention_folate - s_zero_folate
) / s_baseline_folate
intevention_folate_pct_increase_from_zero

scenario      wealth_quintile
intervention  1                  0.761800
              2                  0.644208
              3                  0.540393
              4                  0.467299
              5                  0.205319
dtype: float64

In [57]:
RBC_zero = RBC_baseline / (1 + ((6 / 10) * zero_folate_pct_decrease))
RBC_zero

wealth_quintile
1    901.805420
2    907.704167
3    912.976314
4    916.725226
5    930.418650
dtype: float64

In [58]:
RBC_intervention = RBC_zero * (
    1 + ((6 / 10) * intevention_folate_pct_increase_from_zero)
)
RBC_intervention

scenario      wealth_quintile
intervention  1                  1314.002488
              2                  1258.554271
              3                  1208.996083
              4                  1173.756315
              5                  1045.038130
dtype: float64

In [59]:
def calc_ntd_pr(df, method):
    ln_rbc = np.log(df)
    if method == "daly":
        ln_odds = 1.6563 - 1.2193 * ln_rbc
    elif method == "crider":
        ln_odds = 4.57 - 1.70 * ln_rbc
    p = np.exp(ln_odds)  # TODO: better transformation
    return p

In [60]:
s_ntd_death_or_stillbirth_rate_zero = calc_ntd_pr(RBC_zero, "crider")
10_000 * s_ntd_death_or_stillbirth_rate_zero

wealth_quintile
1    9.141847
2    9.041082
3    8.952506
4    8.890356
5    8.669070
dtype: float64

In [61]:
s_ntd_death_or_stillbirth_rate_intervention = calc_ntd_pr(RBC_intervention, "crider")
10_000 * s_ntd_death_or_stillbirth_rate_intervention

scenario      wealth_quintile
intervention  1                  4.820724
              2                  5.187325
              3                  5.553969
              4                  5.840409
              5                  7.115430
dtype: float64

In [62]:
s_ntd_death_or_stillbirth_count_zero = s_ntd_death_or_stillbirth_rate_zero * s_births
s_ntd_death_or_stillbirth_count_zero

wealth_quintile
1    4414.172404
2    3856.479367
3    3428.292442
4    3210.268928
5    2665.998955
dtype: float64

In [63]:
s_ntd_death_or_stillbirth_count_intervention = (
    s_ntd_death_or_stillbirth_rate_intervention * s_births
)
s_ntd_death_or_stillbirth_count_intervention

scenario      wealth_quintile
intervention  1                  2327.703195
              2                  2212.656819
              3                  2126.849258
              4                  2108.946310
              5                  2188.208165
dtype: float64

In [64]:
ntd_cases_by_scenario = pd.concat(
    [
        s_ntd_death_or_stillbirth_count_zero.rename("value")
        .to_frame()
        .assign(entity="ntd", scenario="zero")
        .set_index(["entity", "scenario"], append=True)
        .value,
        s_ntd_death_or_stillbirth_count.rename("value")
        .to_frame()
        .assign(entity="ntd", scenario="baseline")
        .set_index(["entity", "scenario"], append=True)
        .value,
        *[
            s_ntd_death_or_stillbirth_count_intervention.loc[intervention_scenario]
            .rename("value")
            .to_frame()
            .assign(entity="ntd", scenario=intervention_scenario)
            .set_index(["entity", "scenario"], append=True)
            .value
            for intervention_scenario in intervention_scenarios
        ],
    ]
)
ntd_cases_by_scenario

wealth_quintile  entity  scenario    
1                ntd     zero            4414.172404
2                ntd     zero            3856.479367
3                ntd     zero            3428.292442
4                ntd     zero            3210.268928
                                            ...     
2                ntd     intervention    2212.656819
3                ntd     intervention    2126.849258
4                ntd     intervention    2108.946310
5                ntd     intervention    2188.208165
Name: value, Length: 15, dtype: float64

In [65]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)

In [66]:
# For calculating YLLs
tmrle = vivarium_inputs.get_theoretical_minimum_risk_life_expectancy()
tmrle

,,value
age_start,age_end,
0.00,0.01,89.958040
0.01,0.02,89.975474
0.02,0.03,89.990990
0.03,0.04,89.985077
...,...,...
109.97,109.98,4.509941
109.98,109.99,4.504631
109.99,110.00,4.499321
110.00,125.00,4.494011


In [67]:
# NOTE: Treating stillbirths as a death!
yll_per_ntd = float(tmrle.iloc[0])
yll_per_ntd

89.95803974533831

In [68]:
ylls_by_scenario = ntd_cases_by_scenario * yll_per_ntd
ylls_by_scenario

wealth_quintile  entity  scenario    
1                ntd     zero            397090.296531
2                ntd     zero            346921.324136
3                ntd     zero            308402.467752
4                ntd     zero            288789.499819
                                             ...      
2                ntd     intervention    199046.270081
3                ntd     intervention    191327.190047
4                ntd     intervention    189716.676021
5                ntd     intervention    196846.917043
Name: value, Length: 15, dtype: float64

In [69]:
path = f"./results/{location}/{vehicle}/ylls_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ylls_by_scenario.to_csv(path)